In [2]:
%pip install scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 61.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 62.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 68.5 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 61.9 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
df_clean = pd.read_csv("../data/trips_lyon_cleaned.csv")
print(len(df_clean))

6795


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

cols = ["duration_minutes",
        "distance_km", "battery_start_pct"]
X = StandardScaler().fit_transform(df_clean[cols])
pca = PCA().fit(X)
print(pca.explained_variance_ratio_.round(3))

[0.658 0.333 0.009]


In [5]:
print(pca.explained_variance_ratio_.cumsum().round(3))
print(pd.DataFrame(pca.components_.round(2), columns=cols, index=["PC1","PC2","PC3"]))

[0.658 0.991 1.   ]
     duration_minutes  distance_km  battery_start_pct
PC1              0.71         0.71               0.01
PC2             -0.01        -0.01               1.00
PC3              0.71        -0.71              -0.00


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

features = ["duration_minutes", "distance_km", "vehicle_type"]
X = pd.get_dummies(df_clean[features], drop_first=True)
y = df_clean["price_eur"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42)
model = LinearRegression().fit(X_tr, y_tr)

pred = model.predict(X_te)
print(r2_score(y_te, pred))
print(mean_absolute_error(y_te, pred))
print(dict(zip(X.columns, model.coef_.round(3))))
print(model.intercept_.round(3))

0.9983978736693979
0.048248447204616345
{'duration_minutes': np.float64(0.201), 'distance_km': np.float64(-0.002), 'vehicle_type_mechanical_bike': np.float64(-0.051), 'vehicle_type_scooter': np.float64(0.149)}
1.05


In [8]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_tr, y_tr)
pred_gb = gb.predict(X_te)

print(r2_score(y_te, pred_gb))
print(mean_absolute_error(y_te, pred_gb))
print(dict(zip(X.columns, gb.feature_importances_.round(3))))

0.9982225405565143
0.054636007125425
{'duration_minutes': np.float64(0.998), 'distance_km': np.float64(0.001), 'vehicle_type_mechanical_bike': np.float64(0.0), 'vehicle_type_scooter': np.float64(0.001)}


In [9]:
resid = y - model.predict(X)
print((resid.abs() > 0.1).sum(), "rows off by more than 0.10 €")
cols_show = ["duration_minutes", "distance_km", "vehicle_type",
             "applied_rate_per_min", "price_eur"]
print(df_clean.loc[resid.abs().sort_values(ascending=False).index[:10], cols_show]
      .assign(pred=model.predict(X)[resid.abs().sort_values(ascending=False).index[:10]].round(2)))

1118 rows off by more than 0.10 €
      duration_minutes  distance_km vehicle_type  applied_rate_per_min  \
2688               3.1         2.55          NaN                   0.2   
238                5.4         2.67          NaN                   0.2   
4814               9.3         3.66          NaN                   0.2   
1706               5.2         2.42          NaN                   0.2   
528               14.9         5.07          NaN                   0.2   
5373               3.2         1.82          NaN                   0.2   
6024               6.6         2.75          NaN                   0.2   
3074               9.1         3.40          NaN                   0.2   
5912               3.1         1.71          NaN                   0.2   
2411              10.1         3.62          NaN                   0.2   

      price_eur  pred  
2688       1.82  1.67  
238        2.28  2.13  
4814       3.06  2.91  
1706       2.24  2.09  
528        4.18  4.03  
5373   

In [10]:
print(df_clean["vehicle_type"].value_counts(dropna=False))

fee = (df_clean["price_eur"] - 0.2 * df_clean["duration_minutes"]).round(2)
print(pd.crosstab(df_clean["vehicle_type"].fillna("MISSING"), fee))

vehicle_type
e-bike             3378
scooter            1153
mechanical_bike    1146
NaN                1118
Name: count, dtype: int64
col_0             1.0   1.2
vehicle_type               
MISSING             0  1118
e-bike           3378     0
mechanical_bike  1146     0
scooter             0  1153


In [11]:
X2 = pd.get_dummies(df_clean[features], drop_first=True, dummy_na=True)
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X2, y, test_size=0.2, random_state=42)
m2 = LinearRegression().fit(X_tr2, y_tr2)
print(r2_score(y_te2, m2.predict(X_te2)), mean_absolute_error(y_te2, m2.predict(X_te2)))

1.0 4.341225278041267e-16


In [12]:
df_fix = df_clean.copy()
df_fix["vehicle_type"] = df_fix["vehicle_type"].fillna("scooter")
X3 = pd.get_dummies(df_fix[features], drop_first=True)
X_tr3, X_te3, y_tr3, y_te3 = train_test_split(X3, y, test_size=0.2, random_state=42)
m3 = LinearRegression().fit(X_tr3, y_tr3)
p3 = m3.predict(X_te3)
print(r2_score(y_te3, p3), mean_absolute_error(y_te3, p3))
print(dict(zip(X3.columns, m3.coef_.round(3))), m3.intercept_.round(3))

1.0 8.715128202134783e-16
{'duration_minutes': np.float64(0.2), 'distance_km': np.float64(0.0), 'vehicle_type_mechanical_bike': np.float64(0.0), 'vehicle_type_scooter': np.float64(0.2)} 1.0


In [13]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

# Step 1: Load a copy of the cleaned dataset
try:
    df_pca = df_clean.copy()
except NameError:
    df_pca = pd.read_csv("../data/trips_lyon_cleaned.csv")

# Step 2: Select the permitted numeric features
features = [
    "duration_minutes",
    "distance_km",
    "battery_start_pct"
]

X = df_pca[features].copy()

# Step 3: Inspect rows and missing values
print("Original rows:", len(X))
print("\nMissing values:")
print(X.isna().sum().to_string())

# Step 4: Convert values to numeric and report invalid entries
for col in features:
    X[col] = pd.to_numeric(X[col], errors="coerce")

print("\nMissing or invalid values after conversion:")
print(X.isna().sum().to_string())

# Step 5: Impute missing values without removing rows
print("\nRows requiring imputation:",
      X.isna().any(axis=1).sum())

imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)

print("Rows removed:", 0)
print("Final rows:", len(X_imputed))

# Step 6: Standardize features before PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# Step 7: Fit PCA with all components
pca = PCA()
pca.fit(X_scaled)

# Step 8: Report explained variance
variance = pca.explained_variance_ratio_
cumulative = np.cumsum(variance)

results = pd.DataFrame({
    "Component": [f"PC{i+1}" for i in range(len(variance))],
    "Explained variance": variance,
    "Cumulative variance": cumulative
})

print("\nPCA explained variance:")
print(results.to_string(index=False))

# Step 9: Select enough components to explain >=90%
n_components = np.searchsorted(cumulative, 0.90) + 1

print("\nComponents to keep:", n_components)
print("Variance retained:", cumulative[n_components - 1])

# Step 10: Inspect component loadings
loadings = pd.DataFrame(
    pca.components_.T,
    index=features,
    columns=[f"PC{i+1}" for i in range(len(features))]
)

print("\nPCA loadings:")
print(loadings.to_string())

# Step 11: Interpret retained components
for i in range(n_components):
    weights = loadings.iloc[:, i]
    ranked = weights.abs().sort_values(ascending=False)

    print(f"\nPC{i+1} main contributors:")
    for feature in ranked.index:
        print(f"  {feature}: {weights[feature]:+.4f}")


Original rows: 6795

Missing values:
duration_minutes     0
distance_km          0
battery_start_pct    0

Missing or invalid values after conversion:
duration_minutes     0
distance_km          0
battery_start_pct    0

Rows requiring imputation: 0
Rows removed: 0
Final rows: 6795

PCA explained variance:
Component  Explained variance  Cumulative variance
      PC1            0.657960             0.657960
      PC2            0.333278             0.991238
      PC3            0.008762             1.000000

Components to keep: 2
Variance retained: 0.99123810374884

PCA loadings:
                        PC1       PC2       PC3
duration_minutes   0.707052 -0.008393  0.707112
distance_km        0.707040 -0.010218 -0.707100
battery_start_pct  0.013159  0.999913 -0.001290

PC1 main contributors:
  duration_minutes: +0.7071
  distance_km: +0.7070
  battery_start_pct: +0.0132

PC2 main contributors:
  battery_start_pct: +0.9999
  distance_km: -0.0102
  duration_minutes: -0.0084


In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1. Load the cleaned dataset
df_clean = pd.read_csv("../data/trips_lyon_cleaned.csv")
print(f"Initial rows in df_clean: {len(df_clean)}")

# 2. Select only the numeric feature columns allowed for PCA
pca_cols = ["duration_minutes", "distance_km", "battery_start_pct"]
print("\n--- Missing Values Before PCA ---")
print(df_clean[pca_cols].isna().sum())

# 3. Drop rows with missing values on a copy and report exact counts
df_pca = df_clean[pca_cols].dropna().copy()
rows_dropped = len(df_clean) - len(df_pca)
print(f"\nRows dropped due to missing values in PCA features: {rows_dropped}")
print(f"Remaining rows used for PCA: {len(df_pca)}")

# 4. Standardize the features before fitting PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca)

# 5. Fit PCA and compute variance explained and cumulative variance
pca = PCA()
pca.fit(X_scaled)
explained_var = pca.explained_variance_ratio_
cum_var = np.cumsum(explained_var)

print("\n--- Explained Variance per Component ---")
for i, (var, cvar) in enumerate(zip(explained_var, cum_var), start=1):
    print(f"PC{i}: {var:.4f} ({var*100:.2f}%) | Cumulative: {cvar:.4f} ({cvar*100:.2f}%)")

# 6. Determine how many components are needed to reach at least 90% variance
n_keep = int(np.argmax(cum_var >= 0.90) + 1)
print(f"\nComponents to keep (>= 90% variance threshold): {n_keep}")

# 7. Print component loadings to interpret what each component represents
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f"PC{i}" for i in range(1, len(pca_cols) + 1)],
    index=pca_cols
)
print("\n--- Component Loadings ---")
print(loadings.round(4))

Initial rows in df_clean: 6795

--- Missing Values Before PCA ---
duration_minutes     0
distance_km          0
battery_start_pct    0
dtype: int64

Rows dropped due to missing values in PCA features: 0
Remaining rows used for PCA: 6795

--- Explained Variance per Component ---
PC1: 0.6580 (65.80%) | Cumulative: 0.6580 (65.80%)
PC2: 0.3333 (33.33%) | Cumulative: 0.9912 (99.12%)
PC3: 0.0088 (0.88%) | Cumulative: 1.0000 (100.00%)

Components to keep (>= 90% variance threshold): 2

--- Component Loadings ---
                      PC1     PC2     PC3
duration_minutes   0.7071 -0.0084  0.7071
distance_km        0.7070 -0.0102 -0.7071
battery_start_pct  0.0132  0.9999 -0.0013


In [15]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# Step 1: Load cleaned data without modifying the original
try:
    df = df_clean.copy()
except NameError:
    df = pd.read_csv("../data/trips_lyon_cleaned.csv")

print("Original rows:", len(df))

# Step 2: Select only the permitted features and target
numeric_features = ["duration_minutes", "distance_km"]
categorical_features = ["vehicle_type"]
features = numeric_features + categorical_features
target = "price_eur"

data = df[features + [target]].copy()

# Step 3: Convert numeric columns and report missing values
for col in numeric_features + [target]:
    data[col] = pd.to_numeric(data[col], errors="coerce")

print("\nMissing values before modelling:")
print(data[features + [target]].isna().sum().to_string())

# Step 4: Remove rows with missing target only
missing_target = data[target].isna().sum()
print("\nRows removed (missing target):", missing_target)

data = data.loc[data[target].notna()].copy()
print("Rows remaining:", len(data))

# Step 5: Report missing feature values
print("\nMissing features after target filtering:")
print(data[features].isna().sum().to_string())

print("Rows with any missing feature:",
      data[features].isna().any(axis=1).sum())

# Step 6: Split data into 80% training and 20% testing
X = data[features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))

# Step 7: Prepare numeric and categorical preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Unknown"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Step 8: Create both models
models = {
    "Linear Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]),
    "Gradient Boosting": Pipeline([
        ("preprocessor", preprocessor),
        ("model", GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42
        ))
    ])
}

# Step 9: Train and evaluate both models
results = []

for name, pipeline in models.items():

    pipeline.fit(X_train, y_train)

    train_pred = pipeline.predict(X_train)
    test_pred = pipeline.predict(X_test)

    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)

    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)

    results.append({
        "Model": name,
        "Train R2": train_r2,
        "Test R2": test_r2,
        "Train MAE (EUR)": train_mae,
        "Test MAE (EUR)": test_mae
    })

    print(f"\n{name}")
    print("Train R2:", train_r2)
    print("Test R2:", test_r2)
    print("Train MAE:", train_mae)
    print("Test MAE:", test_mae)

# Step 10: Display model comparison
results_df = pd.DataFrame(results)

print("\n=== MODEL COMPARISON ===")
print(results_df.to_string(index=False))

# Step 11: Extract linear regression coefficients
linear_pipeline = models["Linear Regression"]

linear_features = (
    linear_pipeline.named_steps["preprocessor"]
    .get_feature_names_out()
)

linear_model = linear_pipeline.named_steps["model"]

coefficients = pd.DataFrame({
    "Feature": linear_features,
    "Coefficient": linear_model.coef_
})

print("\n=== LINEAR REGRESSION ===")
print("Intercept:", linear_model.intercept_)
print(coefficients.to_string(index=False))

# Step 12: Extract gradient boosting feature importances
gb_pipeline = models["Gradient Boosting"]

gb_features = (
    gb_pipeline.named_steps["preprocessor"]
    .get_feature_names_out()
)

gb_model = gb_pipeline.named_steps["model"]

importances = pd.DataFrame({
    "Feature": gb_features,
    "Importance": gb_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("\n=== GRADIENT BOOSTING IMPORTANCES ===")
print(importances.to_string(index=False))

# Step 13: Report final row counts
print("\n=== FINAL DATA REPORT ===")
print("Original rows:", len(df))
print("Rows removed (missing target):", missing_target)
print("Final rows:", len(data))
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

# Step 14: Check for unusually high scores
for result in results:
    if result["Test R2"] > 0.99:
        print(
            f"\nWARNING: {result['Model']} has "
            "a near-perfect test R2."
        )
        print(
            "Price may follow a deterministic rule "
            "based on duration and vehicle type."
        )
        print(
            "Check pricing rules and possible leakage "
            "before interpreting this as model success."
        )


Original rows: 6795

Missing values before modelling:
duration_minutes       0
distance_km            0
vehicle_type        1118
price_eur              0

Rows removed (missing target): 0
Rows remaining: 6795

Missing features after target filtering:
duration_minutes       0
distance_km            0
vehicle_type        1118
Rows with any missing feature: 1118

Training rows: 5436
Testing rows: 1359

Linear Regression
Train R2: 1.0
Test R2: 1.0
Train MAE: 4.294577938301682e-15
Test MAE: 4.321291913949395e-15

Gradient Boosting
Train R2: 0.9998810509608277
Test R2: 0.9998267563056036
Train MAE: 0.014300924429119306
Test MAE: 0.01638795469545396

=== MODEL COMPARISON ===
            Model  Train R2  Test R2  Train MAE (EUR)  Test MAE (EUR)
Linear Regression  1.000000 1.000000     4.294578e-15    4.321292e-15
Gradient Boosting  0.999881 0.999827     1.430092e-02    1.638795e-02

=== LINEAR REGRESSION ===
Intercept: 1.1000000000000059
                          Feature   Coefficient
        

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# 1. Load the cleaned dataset without modifying the raw dataframe
df_clean = pd.read_csv("../data/trips_lyon_cleaned.csv")
print(f"Total rows in df_clean: {len(df_clean)}")

# 2. Check and report missing values in target and modelling features before any handling
features = ["duration_minutes", "distance_km", "vehicle_type"]
target = "price_eur"
print("\n--- Missing Values Before Handling ---")
print(df_clean[features + [target]].isna().sum())

# 3. Handle missing values on a copy: fill missing vehicle_type with 'Unknown'
df_model = df_clean[features + [target]].copy()
missing_vt = df_model["vehicle_type"].isna().sum()
df_model["vehicle_type"] = df_model["vehicle_type"].fillna("Unknown")
print(f"\nFilled {missing_vt} missing values in 'vehicle_type' with 'Unknown'. Rows dropped: 0.")

# 4. One-hot encode categorical vehicle_type and separate X and y
X = pd.get_dummies(df_model[features], columns=["vehicle_type"], drop_first=True)
y = df_model[target]
print(f"Final rows used for modelling: {len(X)} | Feature columns: {list(X.columns)}")

# 5. Split data into 80% train and 20% test sets with random_state=42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Train rows: {len(X_train)} | Test rows: {len(X_test)}")

# 6. Fit Linear Regression and evaluate R² and MAE on train and test sets
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_train_pred = lr.predict(X_train)
lr_test_pred = lr.predict(X_test)

print("\n--- Linear Regression Results ---")
print(f"Train R²:  {r2_score(y_train, lr_train_pred):.4f}")
print(f"Test R²:   {r2_score(y_test, lr_test_pred):.4f}")
print(f"Train MAE: {mean_absolute_error(y_train, lr_train_pred):.4f} EUR")
print(f"Test MAE:  {mean_absolute_error(y_test, lr_test_pred):.4f} EUR")
print(f"Intercept: {lr.intercept_:.4f}")
for col, coef in zip(X.columns, lr.coef_):
    print(f"Coefficient [{col}]: {coef:.4f}")

# 7. Fit Gradient Boosting Regressor and evaluate R² and MAE on train and test sets
gbr = GradientBoostingRegressor(random_state=42)
gbr.fit(X_train, y_train)
gbr_train_pred = gbr.predict(X_train)
gbr_test_pred = gbr.predict(X_test)

print("\n--- Gradient Boosting Results ---")
print(f"Train R²:  {r2_score(y_train, gbr_train_pred):.4f}")
print(f"Test R²:   {r2_score(y_test, gbr_test_pred):.4f}")
print(f"Train MAE: {mean_absolute_error(y_train, gbr_train_pred):.4f} EUR")
print(f"Test MAE:  {mean_absolute_error(y_test, gbr_test_pred):.4f} EUR")
for col, imp in zip(X.columns, gbr.feature_importances_):
    print(f"Feature Importance [{col}]: {imp:.4f}")

Total rows in df_clean: 6795

--- Missing Values Before Handling ---
duration_minutes       0
distance_km            0
vehicle_type        1118
price_eur              0
dtype: int64

Filled 1118 missing values in 'vehicle_type' with 'Unknown'. Rows dropped: 0.
Final rows used for modelling: 6795 | Feature columns: ['duration_minutes', 'distance_km', 'vehicle_type_e-bike', 'vehicle_type_mechanical_bike', 'vehicle_type_scooter']
Train rows: 5436 | Test rows: 1359

--- Linear Regression Results ---
Train R²:  1.0000
Test R²:   1.0000
Train MAE: 0.0000 EUR
Test MAE:  0.0000 EUR
Intercept: 1.2000
Coefficient [duration_minutes]: 0.2000
Coefficient [distance_km]: -0.0000
Coefficient [vehicle_type_e-bike]: -0.2000
Coefficient [vehicle_type_mechanical_bike]: -0.2000
Coefficient [vehicle_type_scooter]: -0.0000

--- Gradient Boosting Results ---
Train R²:  0.9999
Test R²:   0.9998
Train MAE: 0.0148 EUR
Test MAE:  0.0169 EUR
Feature Importance [duration_minutes]: 0.9966
Feature Importance [distanc

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# Load cleaned dataset if df_clean is not already in memory
if "df_clean" not in globals():
    df_clean = pd.read_csv("../data/trips_lyon_cleaned.csv")

# Select target and modelling features only
features = ["duration_minutes", "distance_km", "vehicle_type"]
target = "price_eur"
df_model = df_clean[features + [target]].copy()

# Report total rows and missing values per column before handling missingness
print(f"Total rows in df_clean: {len(df_model)}")
print("Missing values per column before handling:")
print(df_model.isna().sum())

# Keep missing vehicle_type entries as an explicit 'unknown' category without dropping rows
unknown_count = df_model["vehicle_type"].isna().sum()
df_model["vehicle_type"] = df_model["vehicle_type"].fillna("unknown")
print(f"Rows assigned to vehicle_type = 'unknown': {unknown_count}")

# One-hot encode vehicle_type and explicitly drop 'e-bike' as the base category
categories = ["e-bike", "mechanical bike", "scooter", "unknown"]
df_model["vehicle_type"] = pd.Categorical(df_model["vehicle_type"], categories=categories)
dummies = pd.get_dummies(df_model["vehicle_type"], prefix="vehicle_type", dtype=float)
dummies = dummies.drop(columns=["vehicle_type_e-bike"])

# Assemble final feature matrix X and target vector y
X = pd.concat([df_model[["duration_minutes", "distance_km"]], dummies], axis=1)
y = df_model[target]

# Split data 80/20 into train and test sets with random_state=42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")

# Define the three models
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression (alpha=1.0)": Ridge(alpha=1.0),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

# Fit models, compute train/test R2 and MAE, and run 5-fold cross-validation
fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    cv_r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
    cv_mae = -cross_val_score(model, X, y, cv=5, scoring="neg_mean_absolute_error")

    print(f"\n--- {name} ---")
    print(f"Train R2: {train_r2:.6f} | Test R2: {test_r2:.6f}")
    print(f"Train MAE: {train_mae:.6f} | Test MAE: {test_mae:.6f}")
    print(f"5-fold CV R2 scores: {np.round(cv_r2, 6)} | Mean: {cv_r2.mean():.6f}")
    print(f"5-fold CV MAE scores: {np.round(cv_mae, 6)} | Mean: {cv_mae.mean():.6f}")

# Print intercepts, coefficients, and implied start fees for Linear and Ridge Regression
for name in ["Linear Regression", "Ridge Regression (alpha=1.0)"]:
    reg = fitted_models[name]
    coefs = pd.Series(reg.coef_, index=X.columns)
    print(f"\n--- {name} Parameters ---")
    print(f"Intercept (e-bike base start fee): {reg.intercept_:.6f}")
    print("Coefficients:")
    print(coefs)

    start_fees = {
        "e-bike": reg.intercept_,
        "mechanical bike": reg.intercept_ + coefs["vehicle_type_mechanical bike"],
        "scooter": reg.intercept_ + coefs["vehicle_type_scooter"],
        "unknown": reg.intercept_ + coefs["vehicle_type_unknown"],
    }
    print("Implied start fees (EUR):")
    for vtype, fee in start_fees.items():
        print(f"  {vtype}: {fee:.6f}")

# Compare Linear Regression and Ridge coefficients directly
lr_coefs = pd.Series(fitted_models["Linear Regression"].coef_, index=X.columns)
ridge_coefs = pd.Series(fitted_models["Ridge Regression (alpha=1.0)"].coef_, index=X.columns)
print("\n--- Coefficient Difference (Ridge - Linear) ---")
print(ridge_coefs - lr_coefs)

# Print feature importances for Gradient Boosting
gb_importances = pd.Series(
    fitted_models["Gradient Boosting"].feature_importances_, index=X.columns
)
print("\n--- Gradient Boosting Feature Importances ---")
print(gb_importances.sort_values(ascending=False))

Total rows in df_clean: 6795
Missing values per column before handling:
duration_minutes       0
distance_km            0
vehicle_type        1118
price_eur              0
dtype: int64
Rows assigned to vehicle_type = 'unknown': 1118
Train rows: 5436, Test rows: 1359


/tmp/ipykernel_7650/2489227058.py:29: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  df_model["vehicle_type"] = pd.Categorical(df_model["vehicle_type"], categories=categories)



--- Linear Regression ---
Train R2: 1.000000 | Test R2: 1.000000
Train MAE: 0.000000 | Test MAE: 0.000000
5-fold CV R2 scores: [1. 1. 1. 1. 1.] | Mean: 1.000000
5-fold CV MAE scores: [0. 0. 0. 0. 0.] | Mean: 0.000000

--- Ridge Regression (alpha=1.0) ---
Train R2: 1.000000 | Test R2: 1.000000
Train MAE: 0.000147 | Test MAE: 0.000147
5-fold CV R2 scores: [1. 1. 1. 1. 1.] | Mean: 1.000000
5-fold CV MAE scores: [0.000146 0.000149 0.000146 0.000147 0.000147] | Mean: 0.000147

--- Gradient Boosting ---
Train R2: 0.999879 | Test R2: 0.999828
Train MAE: 0.014511 | Test MAE: 0.016397
5-fold CV R2 scores: [0.999764 0.999789 0.99965  0.999416 0.999808] | Mean: 0.999685
5-fold CV MAE scores: [0.016633 0.017049 0.01727  0.017204 0.017448] | Mean: 0.017121

--- Linear Regression Parameters ---
Intercept (e-bike base start fee): 1.000000
Coefficients:
duration_minutes                2.000000e-01
distance_km                    -1.043545e-17
vehicle_type_mechanical bike   -2.220446e-16
vehicle_type_s

In [19]:
print(df_model["vehicle_type"].value_counts(dropna=False))

vehicle_type
e-bike             3378
scooter            1153
NaN                1146
unknown            1118
mechanical bike       0
Name: count, dtype: int64


In [20]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# ============================================================
# 1. Load cleaned data without modifying the original
# ============================================================

df_clean = pd.read_csv("../data/trips_lyon_cleaned.csv")
df = df_clean.copy()

features = ["duration_minutes", "distance_km", "vehicle_type"]
target = "price_eur"

print("Original rows:", len(df))

print("\nMissing values before processing:")
print(df[features + [target]].isna().sum())


# ============================================================
# 2. Handle missing vehicle types
# ============================================================

missing_vehicle = df["vehicle_type"].isna().sum()

df["vehicle_type"] = df["vehicle_type"].fillna("unknown")

print("\nVehicle types filled as unknown:", missing_vehicle)

print("\nVehicle type distribution:")
print(df["vehicle_type"].value_counts())


# ============================================================
# 3. Check remaining missing values
# ============================================================

required = ["duration_minutes", "distance_km", "price_eur"]

invalid = df[required].isna().any(axis=1)

print("\nRows removed for missing numeric/target values:",
      invalid.sum())

df = df.loc[~invalid].copy()

print("Final modeling rows:", len(df))


# ============================================================
# 4. Encode vehicle types (e-bike as reference category)
# ============================================================

categories = [
    "e-bike",
    "mechanical_bike",
    "scooter",
    "unknown"
]

observed = set(df["vehicle_type"].unique())

assert observed.issubset(set(categories)), (
    f"Unexpected vehicle categories: {observed - set(categories)}"
)

encoder = OneHotEncoder(
    categories=[categories],
    drop=["e-bike"],
    handle_unknown="ignore",
    sparse_output=False
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            ["duration_minutes", "distance_km"]
        ),
        (
            "vehicle",
            encoder,
            ["vehicle_type"]
        )
    ],
    verbose_feature_names_out=False
)


# ============================================================
# 5. Split data into training and testing sets (80/20)
# ============================================================

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTrain rows:", len(X_train))
print("Test rows:", len(X_test))


# ============================================================
# 6. Define three regression models
# ============================================================

models = {
    "Linear Regression": LinearRegression(),

    "Ridge": Ridge(alpha=1.0),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )
}

fitted_models = {}
results = []


# ============================================================
# 7. Train models and evaluate performance
# ============================================================

for name, estimator in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

    pipeline.fit(X_train, y_train)

    fitted_models[name] = pipeline

    # Predictions
    train_pred = pipeline.predict(X_train)
    test_pred = pipeline.predict(X_test)

    # Training and testing metrics
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)

    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)

    # Five-fold cross-validation on training data
    cv_r2 = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring="r2"
    )

    cv_mae = -cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring="neg_mean_absolute_error"
    )

    results.append({
        "Model": name,
        "Train R2": train_r2,
        "Test R2": test_r2,
        "Train MAE": train_mae,
        "Test MAE": test_mae,
        "CV R2 mean": cv_r2.mean(),
        "CV MAE mean": cv_mae.mean()
    })

    print(f"\n===== {name} =====")

    print(f"Train R2: {train_r2:.8f}")
    print(f"Test R2: {test_r2:.8f}")

    print(f"Train MAE: {train_mae:.8f} EUR")
    print(f"Test MAE: {test_mae:.8f} EUR")

    print("\n5-Fold CV R2 scores:")
    print(cv_r2)

    print("Mean CV R2:", cv_r2.mean())

    print("\n5-Fold CV MAE scores:")
    print(cv_mae)

    print("Mean CV MAE:", cv_mae.mean())


# ============================================================
# 8. Regression coefficients and intercepts
# ============================================================

for name in ["Linear Regression", "Ridge"]:

    pipeline = fitted_models[name]

    estimator = pipeline.named_steps["model"]

    feature_names = pipeline.named_steps[
        "preprocessor"
    ].get_feature_names_out()

    coefficients = pd.Series(
        estimator.coef_,
        index=feature_names
    )

    intercept = estimator.intercept_

    print(f"\n===== {name}: Coefficients =====")

    print("Intercept:", intercept)

    print("\nCoefficients:")
    print(coefficients.to_string())

    # Implied start fees when duration and distance are zero
    print("\nImplied start fees (EUR):")

    print(f"e-bike: {intercept:.8f}")

    for vehicle in categories[1:]:

        key = f"vehicle_type_{vehicle}"

        fee = intercept + coefficients[key]

        print(f"{vehicle}: {fee:.8f}")


# ============================================================
# 9. Compare Ridge and Linear Regression coefficients
# ============================================================

linear = fitted_models[
    "Linear Regression"
].named_steps["model"]

ridge = fitted_models[
    "Ridge"
].named_steps["model"]

print("\n===== Ridge vs Linear Regression =====")

print("\nCoefficient differences (Ridge - Linear):")
print(ridge.coef_ - linear.coef_)

print("\nIntercept difference:")
print(ridge.intercept_ - linear.intercept_)


# ============================================================
# 10. Gradient Boosting feature importances
# ============================================================

gb_pipeline = fitted_models["Gradient Boosting"]

gb_model = gb_pipeline.named_steps["model"]

gb_features = gb_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

importance = pd.Series(
    gb_model.feature_importances_,
    index=gb_features
).sort_values(ascending=False)

print("\n===== Gradient Boosting Feature Importances =====")

print(importance.to_string())


# ============================================================
# 11. Final model comparison
# ============================================================

print("\n===== FINAL MODEL COMPARISON =====")

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))


# ============================================================
# 12. Identify the best model based on test MAE
# ============================================================

best_model = results_df.loc[
    results_df["Test MAE"].idxmin()
]

print("\n===== BEST MODEL =====")

print("Model:", best_model["Model"])
print("Test R2:", best_model["Test R2"])
print("Test MAE:", best_model["Test MAE"])
print("CV R2 mean:", best_model["CV R2 mean"])
print("CV MAE mean:", best_model["CV MAE mean"])


Original rows: 6795

Missing values before processing:
duration_minutes       0
distance_km            0
vehicle_type        1118
price_eur              0
dtype: int64

Vehicle types filled as unknown: 1118

Vehicle type distribution:
vehicle_type
e-bike             3378
scooter            1153
mechanical_bike    1146
unknown            1118
Name: count, dtype: int64

Rows removed for missing numeric/target values: 0
Final modeling rows: 6795

Train rows: 5436
Test rows: 1359

===== Linear Regression =====
Train R2: 1.00000000
Test R2: 1.00000000
Train MAE: 0.00000000 EUR
Test MAE: 0.00000000 EUR

5-Fold CV R2 scores:
[1. 1. 1. 1. 1.]
Mean CV R2: 1.0

5-Fold CV MAE scores:
[3.39475180e-15 3.45445843e-15 3.01179913e-15 1.37250938e-15
 8.28122197e-16]
Mean CV MAE: 2.4123281884603997e-15

===== Ridge =====
Train R2: 0.99999999
Test R2: 0.99999999
Train MAE: 0.00014906 EUR
Test MAE: 0.00015008 EUR

5-Fold CV R2 scores:
[0.99999999 0.99999998 0.99999999 0.99999999 0.99999999]
Mean CV R2: 0.